In [ ]:
import hf_xet
from transformers import pipeline,AutoTokenizer,AutoModel,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding
import torch
from datasets import load_dataset,Dataset
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
import datetime
import numpy as np
import sys
import warnings
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix, classification_report
# from huggingface_hub import notebook_login
np.set_printoptions(threshold=sys.maxsize)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
warnings.filterwarnings('ignore')

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

device = 0 if torch.cuda.is_available() else -1
print(device)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is not available. Possible reasons:")
    print("1. PyTorch was not installed with CUDA support")
    print("2. NVIDIA CUDA toolkit is not installed")
    print("3. Graphics driver issues")
    
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
dataset_dir = "/kaggle/input/nlp-getting-started"
dataset=load_dataset("csv",data_files={'train':f'{dataset_dir}/train.csv','test':f'{dataset_dir}/test.csv'})
dataframes = {split: dataset[split].to_pandas() for split in dataset.keys()}
train_df = dataframes['train']
test_df = dataframes['test']
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df["target"].value_counts(ascending=True).plot.barh()
plt.title("frenquency of classed")

In [ ]:
train_df["words_per_twitter"]=train_df["text"].str.split().apply(len)
test_df["words_per_twitter"]=test_df["text"].str.split().apply(len)
train_df.boxplot("words_per_twitter",by="target",grid=False,showfliers=False,color="black")
print(np.max(train_df['words_per_twitter']))
print(np.max(test_df['words_per_twitter']))

In [ ]:
model_ckpt="distilbert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_batch(texts):
    print(f"Processing batch of {len(texts)} texts")
    encoding = tokenizer(texts, padding='longest', truncation=True, return_tensors="pt")
    return encoding

train_encodings = tokenize_batch(train_df['text'].tolist())
test_encodings = tokenize_batch(test_df['text'].tolist())
print("Train encodings shape:", train_encodings['input_ids'].shape)
print("Test encodings shape:", test_encodings['input_ids'].shape)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model=AutoModel.from_pretrained(model_ckpt).to(device)
num_labels=2
model=(AutoModelForSequenceClassification.from_pretrained(model_ckpt,num_labels=num_labels,ignore_mismatched_sizes=True).to(device))

def compute_metrics(pred):
    labels=pred.label_ids
    preds=pred.predictions.argmax(-1)
    f1=f1_score(labels,preds,average='weighted')
    acc=accuracy_score(labels,preds)
    return {'accuracy':acc,"f1":f1}

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
        }
        if self.labels is not None:
            item['labels']=torch.tensor(self.labels[idx],dtype=torch.long)
        return item
        
    def __len__(self):
        return len(self.encodings['input_ids'])

y_target = train_df['target'].values
full_train_dataset = CustomDataset(train_encodings, y_target)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_train_dataset, [train_size, val_size]
)
batch_size=64
logging_steps=len(train_df)//batch_size
model_name=f"{model_ckpt}-v1"
training_args=TrainingArguments(output_dir=model_name,
                                num_train_epochs=10,
                                load_best_model_at_end=True,
                                metric_for_best_model='f1',
                                greater_is_better=True,
                                learning_rate=2e-5,
                                per_device_train_batch_size=batch_size,
                                per_device_eval_batch_size=batch_size,
                                weight_decay=0.01,
                                eval_strategy='epoch',
                                save_strategy='epoch',
                                disable_tqdm=False,
                                logging_steps=logging_steps,
                                push_to_hub=False,
                                log_level="error",
                                save_total_limit=2, #best and last
                                run_name="v1",
                                report_to="none",  # 确保禁用所有报告
                               )

trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

preds_output = trainer.predict(val_dataset)
y_val_pred = np.argmax(preds_output.predictions, axis=1)

print("Confusion Matrix:")
print(confusion_matrix([val_dataset[i]['labels'].item() for i in range(len(val_dataset))], y_val_pred))
print("\nClassification Report:")
print(classification_report([val_dataset[i]['labels'].item() for i in range(len(val_dataset))], y_val_pred))

test_dataset = CustomDataset(test_encodings)
preds_test=trainer.predict(test_dataset)
y_test_pred=np.argmax(preds_test.predictions,axis=1)

submission_template_path=f'{dataset_dir}/sample_submission.csv'
submission = pd.read_csv(submission_template_path)
print(f"submission: {submission.shape}")
submission['target'] = y_test_pred

print(f"total num: {len(y_test_pred)}")
print(f"positive num: {np.sum(y_test_pred)}")
print(f"negative num: {len(y_test_pred) - np.sum(y_test_pred)}")
print(f"positive ratio: {np.mean(y_test_pred):.3f}")

def save_trained_model(model, tokenizer, base_dir="./trained_models"):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    save_directory = os.path.join(base_dir, f"distilbert_finetuned_{timestamp}")
    os.makedirs(save_directory, exist_ok=True)
    model.save_pretrained(save_directory)
    tokenizer.save_pretrained(save_directory)
    print(f"✅ trained model save to: {save_directory}")
    return save_directory

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"submission_{timestamp}.csv"
if 'Unnamed: 0' in submission.columns:
    print("drop unnamed col...")
    submission = submission.drop('Unnamed: 0', axis=1)
submission.to_csv(submission_filename, index=False)
print(f"✅ submit file saved: {submission_filename}")
model_save_path = save_trained_model(model, tokenizer)